In [6]:
import pandas as pd

import plotly.express as px
import plotly.io as pio

import os

In [7]:
df = pd.read_csv("data.csv", header=None)

for i in range(1, 10):
    df[f"response_{i}"] = (
        df.iloc[:, (i * 3) + 0].astype(str) + "、" +
        df.iloc[:, (i * 3) + 1].astype(str) + "、" +
        df.iloc[:, (i * 3) + 2].astype(str)
    )

drop_id = [0] + [i for i in range(3,30)]    
df = df.drop(df.columns[drop_id], axis=1)
df.columns = ['x', 'y']+[f'response_{i}' for i in range(1,10)]
df.head()

,x,y,response_1,response_2,response_3,response_4,response_5,response_6,response_7,response_8,response_9
0,25,25,Woods、Shadowy、Sunlight,Stones、Monochrome、Array,Starfish、Underwater、Lounging,UGLY、Cybertruck、Tundra,Architecture、Perspective、Curved,Honeybee、Insect、Antennae,Desolate、Sea、Nemo,Mantis、Hunt、Scary,Everest、Peak、Summit
1,31,36,Nature、Green、Light,Rocks、Black、Spacing,Ocean、Star、Light,Trash、Metal、Truck,Confusing、Windows、Curves,Bee、Honey、Legs,Light、Ocean、Sand,Insect、Eyes、Reflection,Peak、Edges、Snow
2,26,36,peaceful、at home、soothing,metalic、dark、alien,plastic、window、see through,elon、tesla、cold,future、museum 、art,bee、pollinator 、protected,ocean、deep、water,intruding、foreign 、hunter,achievement 、success 、goal
3,26,38,Calm、Glow、Peaceful,Solid、Gray、Satisfying,Despair、Glow、Tired,Jagged、Disgust、Choppy,Colorful、Swirly、Curves,Scary、Eyes、Flying,Glow、Depth、Calm,Eyes、Wide、Pointed,Vast、Jagged、Bright
4,33,40,Green、Moss、Camping,Obsidian 、Stone、Grey,Patrick、Starfish、Vein,Cybertruck、Tesla truck 、Boxy,Sky、Pleasing、Airplane,Bee、Barry、Wings,Ocean、Barrier、Light,Serious、Fighting、Movie,Camera、Snow、Division


In [8]:
def formatize(text):
    form = text.split("、")
    return "<br>".join(form)

In [9]:
# Checkerboard 3x3 grid shape definition
def get_checkerboard_shapes(grid_size=16):
    shapes = []
    colors = ["#ffe1e1", '#ffffff']
    for row in range(3):
        for col in range(3):
            x0 = col * grid_size
            y0 = row * grid_size
            color_index = (row + col) % 2
            shapes.append({
                'type': 'rect',
                'xref': 'x',
                'yref': 'y',
                'x0': x0,
                'y0': y0,
                'x1': x0 + grid_size,
                'y1': y0 + grid_size,
                'fillcolor': colors[color_index],
                'line': {'width': 0},
                'layer': 'below'
            })
    return shapes

# Clean fullscreen HTML template with no image
html_template = """
<html>
<head>
  <meta charset="utf-8">
  <style>
    html, body {{
      margin: 0;
      padding: 0;
      width: 100%;
      height: 100%;
    }}
    .plot-container {{
      width: 100%;
      height: 100%;
    }}
  </style>
</head>
<body>
  <div class="plot-container">
    {plot_div}
  </div>
</body>
</html>
"""

# Generate 9 fullscreen interactive plots
for i in range(1, 10):
    col = f'response_{i}'
    filtered_df = df.copy()
    
    filtered_df['response'] = df.loc[filtered_df.index, col].apply(formatize)

    fig = px.scatter(
        filtered_df,
        x='x',
        y='y',
        text='response',
        custom_data=['response'],
        title=f"Interactive Plot – Image {i}",
    )

    fig.update_traces(
        textposition='middle right',textfont=dict(size=10),
        hovertemplate=(
            "%{customdata[0]}<br>"
        )
    )

    fig.update_layout(
        autosize=True,
        xaxis=dict(range=[0, 48], title='E Score'),
        yaxis=dict(range=[0, 48], title='N Score'),
        yaxis_scaleanchor='x',
        plot_bgcolor='white',
        shapes=get_checkerboard_shapes(),
        margin=dict(l=20, r=20, t=60, b=20),
        hoverlabel=dict(
            bgcolor="white",
            font_size=20,
            font_color="black",
            bordercolor="black",
            align='left',
            namelength=0
        )
    )

    fig.update_traces(
        textposition='middle right',textfont=dict(size=15),
    )

    plot_html = pio.to_html(fig, include_plotlyjs='cdn', full_html=False)
    full_html = html_template.format(plot_div=plot_html)

    os.makedirs("htmls", exist_ok=True)

    with open(f"htmls/response_{i}_fullscreen.html", "w", encoding="utf-8") as f:
        f.write(full_html)


In [10]:
# Create a simple HTML index page that links all interactive plots
index_html = """
<html>
<head>
  <style>
    body {{
      font-family: Arial, sans-serif;
      text-align: center;
      background-color: #f8f8f8;
    }}
    h1 {{
      margin-top: 1em;
    }}
    .gallery {{
      display: flex;
      flex-wrap: wrap;
      justify-content: center;
      gap: 20px;
      padding: 2em;
    }}
    .thumb {{
      width: 500px;
      cursor: pointer;
    }}
    .thumb img {{
      width: 100%;
      height: auto;
      border: 2px solid #ccc;
      border-radius: 4px;
    }}
    .thumb-title {{
      margin-top: 0.5em;
      font-weight: bold;
    }}
  </style>
</head>
<body>
  <div class="gallery">
    {thumbnails}
  </div>
</body>
</html>
"""

# Generate thumbnail links
thumbnail_html = ""
for i in range(1, 10):
    image_src = f"images/Images-{i}.png"
    plot_file = f"htmls/response_{i}_fullscreen.html"
    thumbnail_html += f"""
    <div class="thumb" onclick="window.location.href='{plot_file}'">
      <img src="{image_src}" alt="Image {i}">
    </div>
    """

# Fill and save the index HTML
final_html = index_html.format(thumbnails=thumbnail_html)

with open("gallery.html", "w", encoding="utf-8") as f:
    f.write(final_html)

"gallery.html has been generated and links to all 9 interactive plot pages using thumbnail images."



'index.html has been generated and links to all 9 interactive plot pages using thumbnail images.'